### `etd_rk2.ipyng`  
*Created June 12, 2026*

**Description:** This notebook implements the second-order ETD-RK2 method, an exponential time differencing (ETD) integrator for ODE systems of the form 
$$ \frac{du}{dt} = Au + F(u,t) $$ 

where $A$ is an $N \times N$ matrix and $F$ is a non-linear function from $\mathbb{R}^N \times \mathbb{R}$ to $\mathbb{R}^N$. The ETD-RK2 algorithm is as follows: 
\begin{align*}
    a_n &= u_n e^{\Delta t A} + \Delta t \,\big[\varphi_1(\Delta t A) F(u_n,t_n) \big]  \\[5pt]
   u_{n+1} &= a_n + \Delta t \, \varphi_2(\Delta t A) \,\left[F(a_n,t_{n+1}) - F(a_n,t_n) \right]
\end{align*}

where $\varphi_1(\Delta t A)$ and $\varphi_2(\Delta t A)$ are $N \times N$ vectors, and $N(u_n,t_n)$ and $F(a_n,t_n)$ are $N \times 1$ column vectors.

In [3]:
using LinearAlgebra, LaTeXStrings, Random, Printf, NBInclude, Statistics, UnPack
@nbinclude("../../../phi_functions/phi_functions.ipynb")

In [5]:
function etd_rk2(A, f, u0, tspan::NTuple{2,Float64}, p = nothing; dt::Float64)
    """
    PARAMETERS:
    ----------
    A :: N x N constant matrix
    f :: nonlinear function from R^N to R^N 
    u0 :: initial condition (vector of length N)
    dt :: step size 
    p :: parameter for the function `f` (if required)  
    """

    #Input validation
    all(isfinite, tspan) || throw(ArgumentError("Time interval must be finite."))
    tspan[1] < tspan[2] || throw(ArgumentError("`tspan[1]` must be strictly less than `tspan[2]`."))
    isfinite(dt) || throw(ArgumentError("dt must be finite."))
    dt > 0 || throw(ArgumentError("dt must be strictly positive."))
    
    t0, tf = tspan
    M = floor(Int, (tf - t0) / dt)                      #Number of time subintervals
    t = collect(range(t0, t0 + M*dt, length = M + 1))   #Time values at which to record the solution (incl t0)
    u = [zero(u0) for j=1:M+1]                          #Vector (of vectors) to store the solution iterates u_0,…,u_M
    u[1] = u0

    Φ₀ = phis(A*dt, 0)
    Φ₁ = phis(A*dt, 1)
    Φ₂ = phis(A*dt, 2)
    
    for n=1:M      #Compute u[2],...,u[M+1] 
        a = Φ₀ * u[n] + dt * Φ₁ * f(u[n], p, t[n])
        u[n+1] = a + dt * Φ₂ * (f(a, p, t[n+1]) - f(u[n], p, t[n]))
    end 

    return (u = u, t = t, p = p, dt = dt) 
end 

etd_rk2 (generic function with 2 methods)